[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/048.%20The%20Demand%20Forecasting%20-%20Exponential%20Smoothing%20Problem/P48-Tier-1_executed.ipynb)

---


# 48. Demand Forecasting: The Exponential Smoothing Problem

## Tier 1: The Pen-and-Paper Method

This notebook develops single exponential smoothing for a weekly demand series. It first derives the recursive forecast and its expanded form, then measures forecast errors, numerically selects a smoothing parameter, and examines the results through plots and small what-if scenarios.

**Inputs:** observed demand by period, an initial forecast, and a smoothing parameter. **Outputs:** one-step-ahead forecasts, forecast errors and accuracy measures, a numerically selected smoothing parameter, and visualizations. The example uses six weeks of port-terminal demand measured in twenty-foot equivalent units (TEU).

### Model assumptions and scope

- Observations are ordered in time, and the same time interval separates consecutive observations.
- A single smoothing parameter is used throughout the series. The model has no separate trend or seasonal component.
- The initial forecast is specified as part of the model; for the port-terminal example it is 25,000 TEU.
- The parameter is selected by minimizing the sum of squared one-step-ahead forecast errors over the observed data. This is an in-sample criterion, not a guarantee of future forecasting accuracy.
- The numerical search uses the interval $[0.01, 1.0]$. The smoothing model itself is defined for $0 < \alpha \leq 1$.

### Learning sequence

1. Define the recursive forecast and connect it to a weighted history of demand.
2. Implement both forms and verify that they agree.
3. Define forecast errors and accuracy measures, then numerically minimize the sum of squared errors.
4. Interpret the fitted forecasts, error plots, weight distribution, and demand-pattern scenarios.

The parameter $\alpha$ controls responsiveness: a larger value gives more weight to the latest observation, while a smaller value smooths changes more strongly. The weight plot makes this relationship explicit. Because the method has no trend component, a persistent upward or downward pattern can remain difficult to represent.

## Mathematical foundation

A forecast for the next period combines the most recently observed demand with the current forecast. This recursion is useful because it updates the forecast using only the latest observation and the previous forecast, rather than storing and recalculating the entire history.

Let $D_t$ be the observed demand in period $t$, and let $F_t$ be the forecast for period $t$. The forecast for period $t+1$ is

$$
F_{t+1} = \alpha D_t + (1-\alpha)F_t, \qquad 0 < \alpha \leq 1.
$$

Here, $\alpha$ is the smoothing parameter. The coefficients $\alpha$ and $1-\alpha$ are nonnegative and sum to one, so the new forecast is a weighted average of the latest demand and the existing forecast. In the code, this update is implemented by `exponential_smoothing_recursive`.

Repeated substitution expresses the same forecast directly in terms of past observations and the initial forecast:

$$
F_{t+1} = \alpha \sum_{i=1}^{t} (1-\alpha)^{t-i}D_i + (1-\alpha)^t F_1.
$$

In this expression, $D_i$ is the demand in period $i$, and $F_1$ is the initial forecast for period 1. The coefficient on $D_i$ is $\alpha(1-\alpha)^{t-i}$: more recent observations have larger weights, while older observations have geometrically decreasing weights. The initial forecast retains weight $(1-\alpha)^t$. The code implements this expanded expression in `exponential_smoothing_expanded_form`; it is calculated directly to verify the recursive formulation.

For observed periods after the initial forecast, define the forecast error as

$$
E_t = D_t - F_t, \qquad t=2,\ldots,n.
$$

The first observation is not included in the fitting errors because its forecast is the supplied initial value, rather than a forecast generated from earlier observations in the series. The parameter is selected by minimizing the sum of squared errors:

$$
\min_{\alpha \in [0.01,\,1.0]} \quad \sum_{t=2}^{n} E_t^2.
$$

The interval shown is the numerical search interval used in this notebook. Squaring penalizes larger errors more heavily and prevents positive and negative errors from canceling. `objective_function` calculates this sum, and `optimize_alpha` passes it to SciPy's bounded scalar minimizer. The optimizer performs a numerical search; its result should be interpreted in light of the chosen data, initial forecast, objective, and search interval.

Mean squared error (MSE) and mean absolute error (MAE) summarize the same forecast errors on different scales:

$$
\operatorname{MSE} = \frac{1}{n-1}\sum_{t=2}^{n} E_t^2,
\qquad
\operatorname{MAE} = \frac{1}{n-1}\sum_{t=2}^{n} |E_t|.
$$

MSE is in squared demand units (TEU$^2$ here); MAE is in demand units (TEU). Since the number of fitted errors is fixed for a given series, minimizing their sum of squares also minimizes MSE.

The recursive calculation takes $O(n)$ time for $n$ observations. It stores the returned forecast sequence, which requires $O(n)$ space; apart from that output, the update itself needs only a constant amount of working storage. The direct expanded calculation recomputes weighted sums and takes $O(n^2)$ time. Its role here is verification and mathematical interpretation, not a replacement for the recursive calculation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
import seaborn as sns
from typing import List, Tuple

# A consistent plotting style keeps the four diagnostic panels easy to compare.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
class ExponentialSmoothingMathematical:
    """Single exponential smoothing with numerical parameter optimization."""

    def __init__(self, alpha: float = 0.3, initial_forecast: float = None):
        """Initialize the model.

        Args:
            alpha: Smoothing parameter satisfying 0 < alpha <= 1.
            initial_forecast: Forecast for the first period, F_1.
        """
        if not 0 < alpha <= 1:
            raise ValueError("alpha must satisfy 0 < alpha <= 1.")
        if initial_forecast is not None and not np.isfinite(initial_forecast):
            raise ValueError("initial_forecast must be finite when provided.")

        self.alpha = float(alpha)
        self.initial_forecast = initial_forecast
        self.forecasts = []
        self.errors = []

    @staticmethod
    def _validate_demand(demand: List[float]) -> None:
        """Require a nonempty, finite sequence of demand observations."""
        if len(demand) == 0:
            raise ValueError("demand must contain at least one observation.")
        if not np.all(np.isfinite(demand)):
            raise ValueError("demand values must all be finite.")

    def exponential_smoothing_recursive(self, demand: List[float]) -> List[float]:
        """Calculate forecasts with F_(t+1) = alpha * D_t + (1-alpha) * F_t."""
        self._validate_demand(demand)
        if self.initial_forecast is None:
            self.initial_forecast = float(demand[0])

        forecasts = [self.initial_forecast]

        # Each update needs only the latest observation and the preceding forecast.
        for t in range(len(demand) - 1):
            next_forecast = (
                self.alpha * demand[t]
                + (1 - self.alpha) * forecasts[t]
            )
            forecasts.append(next_forecast)

        self.forecasts = forecasts
        return forecasts

    def exponential_smoothing_expanded_form(
        self, demand: List[float]
    ) -> List[float]:
        """Calculate the same forecasts by summing their historical weights."""
        self._validate_demand(demand)
        if self.initial_forecast is None:
            self.initial_forecast = float(demand[0])

        forecasts = [self.initial_forecast]

        # Direct substitution exposes each observation's geometric weight.
        for t in range(1, len(demand)):
            forecast = 0.0
            for i in range(t):
                weight = self.alpha * ((1 - self.alpha) ** (t - 1 - i))
                forecast += weight * demand[i]
            forecast += ((1 - self.alpha) ** t) * self.initial_forecast
            forecasts.append(forecast)

        return forecasts

    def calculate_forecast_errors(
        self, demand: List[float], forecasts: List[float]
    ) -> List[float]:
        """Return errors E_t = D_t - F_t for equally sized sequences."""
        if len(demand) != len(forecasts):
            raise ValueError("demand and forecasts must have the same length.")
        return [actual - forecast for actual, forecast in zip(demand, forecasts)]

    def calculate_mse(self, errors: List[float]) -> float:
        """Return mean squared error; an empty error sequence is undefined."""
        if len(errors) == 0:
            raise ValueError("MSE is undefined for an empty error sequence.")
        return sum(error ** 2 for error in errors) / len(errors)

    def calculate_mae(self, errors: List[float]) -> float:
        """Return mean absolute error; an empty error sequence is undefined."""
        if len(errors) == 0:
            raise ValueError("MAE is undefined for an empty error sequence.")
        return sum(abs(error) for error in errors) / len(errors)

    def objective_function(self, alpha: float, demand: List[float]) -> float:
        """Return the sum of squared one-step-ahead errors for a candidate alpha."""
        if not 0 < alpha <= 1:
            raise ValueError("alpha must satisfy 0 < alpha <= 1.")
        if len(demand) < 2:
            raise ValueError("At least two observations are required for fitting.")

        previous_alpha = self.alpha
        previous_forecasts = self.forecasts
        try:
            self.alpha = float(alpha)
            forecasts = self.exponential_smoothing_recursive(demand)
            errors = self.calculate_forecast_errors(demand[1:], forecasts[1:])
            return sum(error ** 2 for error in errors)
        finally:
            # Optimization evaluates many candidates; retain the model's configured alpha.
            self.alpha = previous_alpha
            self.forecasts = previous_forecasts

    def optimize_alpha(self, demand: List[float]) -> Tuple[float, float]:
        """Numerically minimize the sum of squared errors on [0.01, 1.0]."""
        self._validate_demand(demand)
        if len(demand) < 2:
            raise ValueError("At least two observations are required for fitting.")

        result = minimize_scalar(
            self.objective_function,
            bounds=(0.01, 1.0),
            args=(demand,),
            method="bounded",
        )
        if not result.success:
            raise RuntimeError(f"Alpha optimization failed: {result.message}")

        return float(result.x), float(result.fun)

    def calculate_exponential_weights(self, n_periods: int) -> List[float]:
        """Return demand weights from most recent to progressively older periods."""
        if n_periods < 0:
            raise ValueError("n_periods must be nonnegative.")
        return [
            self.alpha * ((1 - self.alpha) ** i)
            for i in range(n_periods)
        ]

## Port-terminal demand example

The following six observations are weekly demand in TEU. The initial forecast is 25,000 TEU and the first calculation uses $\alpha=0.3$. Forecasts are indexed by the period they predict: the forecast stored at index 0 is $F_1$, and each later forecast is generated from the preceding period's observed demand. Therefore, the first observation is omitted when calculating forecast accuracy.

In [ ]:
# Port-terminal demand forecasting example

weekly_demand = [25000, 28000, 23000, 31000, 27000, 29000]
weeks = list(range(1, len(weekly_demand) + 1))

print("DEMAND FORECASTING: EXPONENTIAL SMOOTHING")
print(f"\nHistorical weekly demand (TEU): {weekly_demand}")
print(f"Number of periods: {len(weekly_demand)}")

# The specified initial forecast anchors the recursion at period 1.
es_model = ExponentialSmoothingMathematical(
    alpha=0.3,
    initial_forecast=25000,
)
forecasts_recursive = es_model.exponential_smoothing_recursive(weekly_demand)
forecasts_expanded = es_model.exponential_smoothing_expanded_form(weekly_demand)
errors = es_model.calculate_forecast_errors(weekly_demand, forecasts_recursive)

# Period 1 is excluded because it uses the supplied initial forecast.
valid_errors = errors[1:]
mse = es_model.calculate_mse(valid_errors)
mae = es_model.calculate_mae(valid_errors)

print("\nRECURSIVE FORMULATION RESULTS")
print("Week | Demand (TEU) | Forecast (TEU) | Error (TEU)")
print("-" * 55)
for week, demand, forecast, error in zip(
    weeks, weekly_demand, forecasts_recursive, errors
):
    print(f"{week:4d} | {demand:12.0f} | {forecast:14.0f} | {error:11.0f}")

print("\nACCURACY METRICS, PERIODS 2 THROUGH 6")
print(f"Mean squared error (TEU^2): {mse:,.0f}")
print(f"Mean absolute error (TEU): {mae:,.0f}")
print(f"Smoothing parameter (alpha): {es_model.alpha:.1f}")

# Agreement verifies that direct substitution and recursion encode the same model.
max_diff = max(
    abs(recursive - expanded)
    for recursive, expanded in zip(forecasts_recursive, forecasts_expanded)
)
print("\nFORMULATION VERIFICATION")
print(f"Maximum absolute difference: {max_diff:.12g}")
print(f"Formulations agree within numerical tolerance: {np.isclose(max_diff, 0.0, atol=1e-10)}")
assert np.allclose(forecasts_recursive, forecasts_expanded, rtol=1e-12, atol=1e-10)

## Numerical selection of the smoothing parameter

The recursive formula defines a forecast sequence for any admissible $\alpha$. To select a value using the observed data, the notebook evaluates the sum of squared errors for candidate values and uses a bounded scalar numerical minimizer over $[0.01,1.0]$. The initial forecast remains fixed at 25,000 TEU, and the error periods remain 2 through 6. The resulting parameter is specific to this fitting criterion and sample; it is not evidence that the same value will be best for future data.

In [ ]:
print("PARAMETER OPTIMIZATION")

optimal_alpha, min_objective = es_model.optimize_alpha(weekly_demand)

# Apply the selected parameter to the same demand series and initial forecast.
es_optimal = ExponentialSmoothingMathematical(
    alpha=optimal_alpha,
    initial_forecast=25000,
)
optimal_forecasts = es_optimal.exponential_smoothing_recursive(weekly_demand)
optimal_errors = es_optimal.calculate_forecast_errors(
    weekly_demand,
    optimal_forecasts,
)
optimal_valid_errors = optimal_errors[1:]
optimal_mse = es_optimal.calculate_mse(optimal_valid_errors)
optimal_mae = es_optimal.calculate_mae(optimal_valid_errors)

print(f"Configured alpha = {es_model.alpha:.3f}: MSE = {mse:,.0f} TEU^2, MAE = {mae:,.0f} TEU")
print(f"Selected alpha = {optimal_alpha:.6f}: MSE = {optimal_mse:,.0f} TEU^2, MAE = {optimal_mae:,.0f} TEU")
print(f"Minimum sum of squared errors: {min_objective:,.0f} TEU^2")

# A finite, nonzero baseline permits an interpretable percentage comparison.
if not np.isclose(mse, 0.0):
    mse_reduction = (mse - optimal_mse) / mse * 100
    print(f"MSE reduction relative to alpha = 0.3: {mse_reduction:.2f}%")
if not np.isclose(mae, 0.0):
    mae_reduction = (mae - optimal_mae) / mae * 100
    print(f"MAE reduction relative to alpha = 0.3: {mae_reduction:.2f}%")

# This grid shows how the error measures vary across selected parameter values.
alpha_values = [0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9]
alpha_results = []

for alpha in alpha_values:
    es_test = ExponentialSmoothingMathematical(
        alpha=alpha,
        initial_forecast=25000,
    )
    test_forecasts = es_test.exponential_smoothing_recursive(weekly_demand)
    test_errors = es_test.calculate_forecast_errors(
        weekly_demand,
        test_forecasts,
    )[1:]
    alpha_results.append(
        {
            "alpha": alpha,
            "mse": es_test.calculate_mse(test_errors),
            "mae": es_test.calculate_mae(test_errors),
        }
    )

print("\nSENSITIVITY ANALYSIS")
print("Alpha | MSE (TEU^2) | MAE (TEU)")
print("-" * 36)
for result in alpha_results:
    print(
        f"{result['alpha']:5.1f} | "
        f"{result['mse']:11.0f} | "
        f"{result['mae']:9.0f}"
    )

## Visual diagnostics

The four panels compare observed demand with forecasts, show the signed errors used in fitting, display the geometric weights assigned to recent and older observations, and plot the error measures across the sensitivity grid. Error bars are shown for periods 2 onward because period 1 is the supplied initial forecast. In the weight panel, lag 0 means the most recent observed demand; increasing lag means an older observation.

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle(
    "Demand Forecasting: Single Exponential Smoothing",
    fontsize=16,
    fontweight="bold",
)

# Both forecast lines use the same initial value and differ only in alpha.
ax1.plot(
    weeks,
    weekly_demand,
    "o-",
    linewidth=2,
    markersize=8,
    label="Actual demand",
    color="blue",
)
ax1.plot(
    weeks,
    forecasts_recursive,
    "s--",
    linewidth=2,
    markersize=6,
    label=f"Forecast (alpha={es_model.alpha:.1f})",
    color="red",
)
ax1.plot(
    weeks,
    optimal_forecasts,
    "^:",
    linewidth=2,
    markersize=6,
    label=f"Selected alpha ({optimal_alpha:.3f})",
    color="green",
)
ax1.set_xlabel("Week")
ax1.set_ylabel("Demand (TEU)")
ax1.set_title("Actual and Forecast Demand")
ax1.legend()
ax1.grid(True, alpha=0.3)

error_weeks = weeks[1:]
ax2.bar(
    error_weeks,
    errors[1:],
    alpha=0.7,
    label=f"Errors (alpha={es_model.alpha:.1f})",
    color="orange",
)
ax2.bar(
    error_weeks,
    optimal_errors[1:],
    alpha=0.5,
    label=f"Errors (alpha={optimal_alpha:.3f})",
    color="purple",
)
ax2.axhline(y=0, color="black", linestyle="-", linewidth=0.8)
ax2.set_xlabel("Week")
ax2.set_ylabel("Forecast error (TEU)")
ax2.set_title("One-Step-Ahead Forecast Errors")
ax2.legend()
ax2.grid(True, alpha=0.3)

# Lag zero is the latest demand observation and receives weight alpha.
weight_lags = list(range(10))
weights_original = es_model.calculate_exponential_weights(len(weight_lags))
es_optimal_weights = ExponentialSmoothingMathematical(alpha=optimal_alpha)
weights_optimal = es_optimal_weights.calculate_exponential_weights(
    len(weight_lags)
)
ax3.plot(
    weight_lags,
    weights_original,
    "o-",
    linewidth=2,
    markersize=6,
    label=f"alpha={es_model.alpha:.1f}",
    color="red",
)
ax3.plot(
    weight_lags,
    weights_optimal,
    "s--",
    linewidth=2,
    markersize=6,
    label=f"alpha={optimal_alpha:.3f}",
    color="green",
)
ax3.set_xlabel("Lag (periods before the latest observation)")
ax3.set_ylabel("Demand weight")
ax3.set_title("Geometric Weights on Historical Demand")
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_yscale("log")

alpha_plot_values = [result["alpha"] for result in alpha_results]
mse_plot_values = [result["mse"] for result in alpha_results]
mae_plot_values = [result["mae"] for result in alpha_results]

ax4_twin = ax4.twinx()
line1 = ax4.plot(
    alpha_plot_values,
    mse_plot_values,
    "o-",
    linewidth=2,
    markersize=6,
    label="MSE",
    color="blue",
)
line2 = ax4_twin.plot(
    alpha_plot_values,
    mae_plot_values,
    "s--",
    linewidth=2,
    markersize=6,
    label="MAE",
    color="red",
)
ax4.axvline(
    x=optimal_alpha,
    color="green",
    linestyle=":",
    linewidth=2,
    label=f"Numerically selected alpha={optimal_alpha:.3f}",
)
ax4.set_xlabel("Smoothing parameter (alpha)")
ax4.set_ylabel("Mean squared error (TEU^2)", color="blue")
ax4_twin.set_ylabel("Mean absolute error (TEU)", color="red")
ax4.set_title("Error Measures Across the Sensitivity Grid")

# The two error metrics use different units and therefore have separate axes.
lines = line1 + line2 + [ax4.lines[-1]]
labels = [line.get_label() for line in lines]
ax4.legend(lines, labels, loc="upper left")
ax4.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## Extended calculations and error interpretation

A single forecast update can be expanded into its weighted contributions. The calculation below uses the forecast for week 4: it combines demand observed through week 3 with the initial forecast. The error table then separates signed error, squared error, absolute error, and absolute percentage error. Signed errors retain direction; the other measures summarize error magnitude.

In [ ]:
print("MATHEMATICAL ANALYSIS: EXTENDED CALCULATIONS")

# F_4 is generated from week 3 demand and the forecast for week 3.
print("\nRECURSIVE CALCULATION: WEEK 4 FORECAST")
print("F_4 = alpha * D_3 + (1 - alpha) * F_3")
print(
    f"F_4 = {es_model.alpha:.1f} * {weekly_demand[2]} + "
    f"{1 - es_model.alpha:.1f} * {forecasts_recursive[2]:.0f}"
)
print(
    f"F_4 = {es_model.alpha * weekly_demand[2]:.0f} + "
    f"{(1 - es_model.alpha) * forecasts_recursive[2]:.0f}"
)
print(f"F_4 = {forecasts_recursive[3]:.0f}")

# The expanded expression makes the contribution of each observation visible.
print("\nEXPANDED CALCULATION: WEEK 4 FORECAST")
print("F_4 = alpha * sum((1 - alpha)^(3 - i) * D_i) + (1 - alpha)^3 * F_1")
expanded_calc = 0.0
for i in range(3):
    weight = es_model.alpha * ((1 - es_model.alpha) ** (2 - i))
    contribution = weight * weekly_demand[i]
    expanded_calc += contribution
    print(
        f"Demand in week {i + 1}: "
        f"weight {weight:.3f} * {weekly_demand[i]} = {contribution:.0f} TEU"
    )

initial_weight = (1 - es_model.alpha) ** 3
initial_contribution = initial_weight * es_model.initial_forecast
expanded_calc += initial_contribution
print(
    f"Initial forecast: weight {initial_weight:.3f} * "
    f"{es_model.initial_forecast} = {initial_contribution:.0f} TEU"
)
print(
    f"Expanded forecast = {expanded_calc:.0f} TEU "
    f"(recursive forecast = {forecasts_recursive[3]:.0f} TEU)"
)
assert np.isclose(expanded_calc, forecasts_recursive[3], rtol=1e-12, atol=1e-8)

print("\nDETAILED ACCURACY METRICS, PERIODS 2 THROUGH 6")
squared_errors = [error ** 2 for error in valid_errors]
absolute_errors = [abs(error) for error in valid_errors]
percent_errors = [
    abs(error) / weekly_demand[week - 1] * 100
    for week, error in zip(error_weeks, valid_errors)
]

print("Week | Error (TEU) | Squared error (TEU^2) | Absolute error (TEU) | Absolute percentage error")
print("-" * 91)
for i, week in enumerate(error_weeks):
    print(
        f"{week:4d} | {valid_errors[i]:11.0f} | "
        f"{squared_errors[i]:21.0f} | {absolute_errors[i]:20.0f} | "
        f"{percent_errors[i]:24.1f}%"
    )

sum_squared_errors = sum(squared_errors)
mean_squared_error = sum_squared_errors / len(squared_errors)
root_mean_squared_error = np.sqrt(mean_squared_error)
mean_absolute_error = sum(absolute_errors) / len(absolute_errors)
mean_absolute_percentage_error = sum(percent_errors) / len(percent_errors)

print("\nSummary statistics")
print(f"Sum of squared errors (TEU^2): {sum_squared_errors:,.0f}")
print(f"Mean squared error (TEU^2): {mean_squared_error:,.0f}")
print(f"Root mean squared error (TEU): {root_mean_squared_error:,.0f}")
print(f"Mean absolute error (TEU): {mean_absolute_error:,.0f}")
print(f"Mean absolute percentage error: {mean_absolute_percentage_error:.1f}%")

## What-if analysis: demand patterns

The scenarios below keep the single exponential smoothing method and compare a fixed $\alpha=0.3$ with a value selected separately for each scenario using the same numerical objective. The initial forecast for each scenario is its first demand observation. These short, constructed series illustrate how the fitted parameter and error can depend on the observed pattern; they do not establish a general rule that a particular pattern always requires a particular parameter. Single exponential smoothing has no trend component, so a trend scenario also illustrates a structural limitation of the model.

In [ ]:
print("WHAT-IF ANALYSIS: DIFFERENT DEMAND PATTERNS")

stable_demand = [25000, 25500, 24800, 25200, 25100, 25300]
volatile_demand = [25000, 35000, 15000, 40000, 12000, 38000]
trending_demand = [20000, 22000, 24000, 26000, 28000, 30000]

scenarios = [
    ("Stable", stable_demand),
    ("Volatile", volatile_demand),
    ("Trending", trending_demand),
]
scenario_results = []

for scenario_name, scenario_demand in scenarios:
    initial_forecast = scenario_demand[0]

    es_scenario = ExponentialSmoothingMathematical(
        alpha=0.3,
        initial_forecast=initial_forecast,
    )
    scenario_forecasts = es_scenario.exponential_smoothing_recursive(
        scenario_demand
    )
    scenario_errors = es_scenario.calculate_forecast_errors(
        scenario_demand,
        scenario_forecasts,
    )[1:]
    scenario_mse = es_scenario.calculate_mse(scenario_errors)

    # Each scenario uses the same objective and bounded search as the main example.
    optimal_alpha_scenario, _ = es_scenario.optimize_alpha(scenario_demand)
    es_optimal_scenario = ExponentialSmoothingMathematical(
        alpha=optimal_alpha_scenario,
        initial_forecast=initial_forecast,
    )
    optimal_forecasts_scenario = es_optimal_scenario.exponential_smoothing_recursive(
        scenario_demand
    )
    optimal_errors_scenario = es_optimal_scenario.calculate_forecast_errors(
        scenario_demand,
        optimal_forecasts_scenario,
    )[1:]
    optimal_mse_scenario = es_optimal_scenario.calculate_mse(
        optimal_errors_scenario
    )

    improvement = (
        (scenario_mse - optimal_mse_scenario) / scenario_mse * 100
        if not np.isclose(scenario_mse, 0.0)
        else np.nan
    )
    scenario_results.append(
        {
            "scenario": scenario_name,
            "demand": scenario_demand,
            "mse_original": scenario_mse,
            "optimal_alpha": optimal_alpha_scenario,
            "mse_optimal": optimal_mse_scenario,
            "improvement": improvement,
        }
    )

print("Scenario | Fixed-alpha MSE (TEU^2) | Selected alpha | Selected MSE (TEU^2) | MSE reduction")
print("-" * 87)
for result in scenario_results:
    improvement_text = (
        f"{result['improvement']:.1f}%"
        if np.isfinite(result["improvement"])
        else "undefined"
    )
    print(
        f"{result['scenario']:8s} | "
        f"{result['mse_original']:22.0f} | "
        f"{result['optimal_alpha']:14.3f} | "
        f"{result['mse_optimal']:21.0f} | "
        f"{improvement_text:13s}"
    )

print("\nInterpretation")
print(
    "Compare the scenario-specific selected alpha values and error measures "
    "with the observed demand sequences."
)
print(
    "A lower in-sample error describes fit to these observations; it does not "
    "by itself establish better performance on future observations."
)
print(
    "The trending sequence is included to show that adjusting alpha does not "
    "add a separate trend component to the model."
)

## Summary and conclusions

Single exponential smoothing updates a forecast by combining the latest observation with the preceding forecast:

$$
F_{t+1} = \alpha D_t + (1-\alpha)F_t.
$$

Repeated substitution gives a weighted sum of historical observations and the initial forecast. The demand weights decrease geometrically with age, while $\alpha$ controls the trade-off between responsiveness to recent observations and smoothing of short-term variation. The recursive and expanded implementations represent the same forecast; the notebook checks their numerical agreement.

For the observed periods after the initial forecast, the fitting objective is the sum of squared one-step-ahead errors. The bounded numerical search selects a parameter for this data, initialization, objective, and search interval. MSE and MAE describe error magnitude in complementary ways, while the sensitivity grid and scenarios show how those measures vary with the parameter and demand pattern.

The recursive update requires linear time in the number of observations. It returns and stores the forecast sequence, so total storage includes that sequence; the update itself uses constant working storage. The expanded calculation is useful for understanding the weights and verifying the recursion, but direct repeated summation requires quadratic time.

### Limitations and scope

The method uses one fixed smoothing parameter and has no explicit trend or seasonal component. Its in-sample optimization does not guarantee future accuracy, and conclusions depend on the chosen initial forecast and error criterion. These properties define the scope of this Tier 1 model and motivate studying adaptive smoothing and richer demand structures in subsequent material.